# Fast DeltaNet kernel probe (M8)

The fast-kernel check recorded a 1.3x speed-up with a maximum log-prob difference of exactly zero.
The first version of this probe (2026-11-11, transformers 5.20.2, fla 0.6.0) found why: in
`modeling_qwen3_5`, `torch_chunk_gated_delta_rule` is decorated with
`use_kernel_func_from_hub_with_fallback("chunk_gated_delta_rule", "fla")`. Once fla is
installed, the "torch" name already dispatches to the fla kernel. On random tensors the two names
gave bit-identical outputs at the same speed (1.01x), so the check compared fla with itself.

This version separates them:

1. **Unwrap.** Recovers the undecorated torch function and prints the decorator's source.
2. **Kernel parity, no model.** The pure torch rule, the fla kernel and the decorated name on
   the same random inputs. The run stops here if the torch rule cannot be separated.
3. **Model, needs GPU T4 x2.** Three passes with the module-level name bound to the pure torch
   rule, to fla, and to the decorated original. Each input is warmed up before it is timed, and
   each timing is the faster of two forwards. Compares the first DeltaNet layer's output and the
   last-position log-probs against the torch pass, then profiles a 7k-token forward with torch
   and with fla.

Writes `/kaggle/working/m8_result.json` and `/kaggle/working/m8_profile_{torch,fla}.txt`.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'
FRESH = True
REQUIRE = [('sleep/lm.py', 'chat_template_kwargs')]

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

if FRESH and GIT_URL and os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO], check=True)
    print(f'removed {REPO} so the clone is current')

if not looks_like_source(REPO) and GIT_URL:
    r = subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)

if not looks_like_source(REPO):
    print('no clone; falling back to attached inputs')
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)

assert looks_like_source(REPO), (
    'No source found. Either (a) set GIT_URL above, or (b) run scripts/package_source.py '
    'locally, upload the zip as a Kaggle Dataset, and attach it via Add Input. '
    f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)

_h = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True, cwd=REPO)
print('source commit:', _h.stdout.strip() or 'unknown (not a git checkout)')

_stale = [f for f, token in REQUIRE
          if not os.path.exists(f) or token not in open(f, encoding='utf-8').read()]
assert not _stale, (
    f'This checkout is older than the seed support these cells need: {_stale}. '
    'It came from an attached dataset or a leftover /kaggle/working/myrios rather than a '
    'fresh clone. Detach any source dataset under Add Input, keep only the runs_declare.zip '
    'dataset, and re-run this cell with FRESH = True.')
print('source is current for the seed grid')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass

exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Installs the kernels before any model code is imported.


In [ ]:
import os, sys, subprocess
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.46.1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'flash-linear-attention', 'fla-core'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'list', '--format=freeze'], check=False,
               stdout=open('/kaggle/working/pip_freeze.txt', 'w'))
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

CFG = 'configs/kaggle_declare_27b.yaml'
MODEL = 'Qwen/Qwen3.8-27B'
from common.io import load_config
_c = load_config(CFG)
assert _c['model']['base'] == MODEL


## Loader


In [ ]:
import torch
from transformers import AutoConfig
from transformers.models.auto.modeling_auto import MODEL_FOR_CAUSAL_LM_MAPPING
print('gpus', torch.cuda.device_count())
cfg_hf = AutoConfig.from_pretrained(MODEL)
LOADER = 'causal' if type(cfg_hf) in MODEL_FOR_CAUSAL_LM_MAPPING else 'image_text'
print('loader', LOADER)


## Probe


In [ ]:
PROBE_SRC = r'''import sys, json, time, inspect, importlib, torch
sys.path.insert(0, '/kaggle/working/myrios')
import transformers, fla, triton
res = {'versions': {'transformers': transformers.__version__, 'fla': getattr(fla, '__version__', '?'),
                    'triton': triton.__version__, 'torch': torch.__version__},
       'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
mod = importlib.import_module('transformers.models.qwen3_5.modeling_qwen3_5')
from fla.ops.gated_delta_rule import chunk_gated_delta_rule as fla_rule
decorated = mod.torch_chunk_gated_delta_rule


def unwrap(f):
    seen = set()
    while id(f) not in seen:
        seen.add(id(f))
        if hasattr(f, '__wrapped__'):
            f = f.__wrapped__
            continue
        inner = [c.cell_contents for c in (getattr(f, '__closure__', None) or ())
                 if callable(getattr(c, 'cell_contents', None))
                 and getattr(c.cell_contents, '__name__', '') == 'torch_chunk_gated_delta_rule']
        if inner and inner[0] is not f:
            f = inner[0]
            continue
        break
    return f


torch_rule = unwrap(decorated)
res['decorated_repr'] = repr(decorated)
res['unwrapped_repr'] = repr(torch_rule)
res['unwrapped_differs'] = torch_rule is not decorated
dec = None
for name in (decorated.__module__, 'transformers.integrations.hub_kernels', 'transformers.integrations'):
    try:
        dec = getattr(importlib.import_module(name), 'use_kernel_func_from_hub_with_fallback')
        break
    except Exception:
        pass
res['decorator_source'] = inspect.getsource(dec).splitlines()[:80] if dec else 'not found'
res['binding_lines'] = [l.strip() for l in inspect.getsource(mod).splitlines()
                        if 'chunk_gated_delta_rule' in l or 'is_fast_path_available' in l]
print(json.dumps(res, indent=1), flush=True)


def save():
    json.dump(res, open('/kaggle/working/m8_result.json', 'w'), indent=1)


def timed(fn, *a, reps=3, **kw):
    fn(*a, **kw)
    torch.cuda.synchronize()
    t = time.time()
    for _ in range(reps):
        o = fn(*a, **kw)
    torch.cuda.synchronize()
    return o, (time.time() - t) / reps


torch.manual_seed(0)
B, T, H, K, V = 1, 4096, 16, 128, 128
q = torch.randn(B, T, H, K, device='cuda:0', dtype=torch.bfloat16)
k = torch.randn(B, T, H, K, device='cuda:0', dtype=torch.bfloat16)
v = torch.randn(B, T, H, V, device='cuda:0', dtype=torch.bfloat16)
g = torch.nn.functional.logsigmoid(torch.randn(B, T, H, device='cuda:0', dtype=torch.float32))
beta = torch.rand(B, T, H, device='cuda:0', dtype=torch.bfloat16)
kw = dict(g=g, beta=beta, initial_state=None, output_final_state=False, use_qk_l2norm_in_kernel=True)
outs = {}
res['kernel_parity'] = {'shape': [B, T, H, K, V]}
for name, fn in (('torch', torch_rule), ('fla', fla_rule), ('decorated', decorated)):
    (o, _), secs = timed(fn, q, k, v, **kw)
    outs[name] = o.float()
    res['kernel_parity'][f'{name}_s'] = secs
for name in ('fla', 'decorated'):
    d = (outs['torch'] - outs[name]).abs()
    res['kernel_parity'][f'{name}_vs_torch'] = {'exact_equal': bool(torch.equal(outs['torch'], outs[name])),
                                               'max_abs_diff': d.max().item(), 'mean_abs_diff': d.mean().item()}
res['kernel_parity']['decorated_equals_fla'] = bool(torch.equal(outs['decorated'], outs['fla']))
res['kernel_parity']['max_abs_out'] = outs['torch'].abs().max().item()
print('kernel parity', json.dumps(res['kernel_parity'], indent=1), flush=True)
del q, k, v, g, beta, outs
torch.cuda.empty_cache()
save()

if not res['unwrapped_differs'] or res['kernel_parity']['fla_vs_torch']['exact_equal']:
    res['model'] = 'skipped: the pure torch rule could not be separated from the fla dispatch'
    save()
    print(res['model'])
    raise SystemExit(0)

if torch.cuda.device_count() < 2:
    res['model'] = 'skipped: needs GPU T4 x2 (Session options -> Accelerator)'
    save()
    print(res['model'])
    raise SystemExit(0)

from transformers import AutoConfig
from transformers.models.auto.modeling_auto import MODEL_FOR_CAUSAL_LM_MAPPING
from common.io import load_config
from sleep.lm import load_backbone, chat_text
from declare.elicit import register_repeat
CFG = sys.argv[1]
hf = AutoConfig.from_pretrained(load_config(CFG)['model']['base'])
loader = 'causal' if type(hf) in MODEL_FOR_CAUSAL_LM_MAPPING else 'image_text'
model, tokenizer = load_backbone(load_config(CFG, {'model': {'loader': loader}}), attn_implementation=register_repeat())
layers = [m for m in model.modules() if 'GatedDeltaNet' in type(m).__name__]
res['deltanet_layers'] = len(layers)
res['deltanet_class'] = type(layers[0]).__name__ if layers else None
save()
assert layers, 'no GatedDeltaNet module found'
calls = {'n': 0}


def counted(fn):
    def wrapper(*a, chunk_size=None, **kw):
        calls['n'] += 1
        if fn is fla_rule or chunk_size is None:
            return fn(*a, **kw)
        return fn(*a, chunk_size=chunk_size, **kw)
    return wrapper


RULES = {'torch': counted(torch_rule), 'fla': counted(fla_rule), 'decorated': counted(decorated)}
captured = {}


def hook(m, i, o):
    captured['out'] = (o[0] if isinstance(o, tuple) else o).detach().float().cpu()


layers[0].register_forward_hook(hook)
dev = next(model.parameters()).device
short = chat_text(tokenizer, None, 'Answer with only Yes or No. Is Paris the capital of France?')
filler = ' '.join(f'Item {i} is stored in box {i * 7 % 101}.' for i in range(1200))
long_text = chat_text(tokenizer, None, filler + ' Answer with only Yes or No. Is box 7 listed?')
inputs = {'short': tokenizer([short], return_tensors='pt')['input_ids'].to(dev),
          'long': tokenizer([long_text], return_tensors='pt')['input_ids'][:, -7000:].to(dev)}


def forward(ids):
    torch.cuda.synchronize()
    t = time.time()
    with torch.no_grad():
        lp = torch.log_softmax(model(input_ids=ids, logits_to_keep=1).logits[0, -1].float(), -1).cpu()
    torch.cuda.synchronize()
    return lp, time.time() - t


runs = {}
for label in ('torch', 'fla', 'decorated'):
    mod.torch_chunk_gated_delta_rule = RULES[label]
    for ids in inputs.values():
        forward(ids)
    for name, ids in inputs.items():
        calls['n'] = 0
        lp, secs = forward(ids)
        lp2, secs2 = forward(ids)
        runs[(label, name)] = {'lp': lp, 'secs': min(secs, secs2), 'calls': calls['n'] // 2,
                               'layer0': captured['out'], 'repeat_equal': bool(torch.equal(lp, lp2))}
        print(f'{label:9s} {name:5s} {secs:7.2f} s / {secs2:7.2f} s  rule calls {calls["n"] // 2}', flush=True)
        torch.cuda.empty_cache()
mod.torch_chunk_gated_delta_rule = decorated

cmp = {}
for label in ('torch', 'fla', 'decorated'):
    for name in inputs:
        a, b = runs[('torch', name)], runs[(label, name)]
        cmp[f'{label}/{name}'] = {
            'seconds': b['secs'],
            'rule_calls_per_forward': b['calls'],
            'repeat_bit_identical': b['repeat_equal'],
            'layer0_max_abs_diff_vs_torch': (a['layer0'] - b['layer0']).abs().max().item(),
            'layer0_exact_equal_to_torch': bool(torch.equal(a['layer0'], b['layer0'])),
            'logprob_max_abs_diff_vs_torch': (a['lp'] - b['lp']).abs().max().item(),
            'same_top1_as_torch': a['lp'].argmax().item() == b['lp'].argmax().item(),
            'speedup_vs_torch': a['secs'] / b['secs']}
res['model'] = cmp
print(json.dumps(cmp, indent=1), flush=True)
save()

from torch.profiler import profile, ProfilerActivity
for label in ('torch', 'fla'):
    mod.torch_chunk_gated_delta_rule = RULES[label]
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        forward(inputs['long'])
    table = prof.key_averages().table(sort_by='cuda_time_total', row_limit=30)
    open(f'/kaggle/working/m8_profile_{label}.txt', 'w').write(table)
    print(label)
    print(table)
mod.torch_chunk_gated_delta_rule = decorated
save()
print('wrote /kaggle/working/m8_result.json')
'''
open('/kaggle/working/m8_probe.py', 'w').write(PROBE_SRC)
run(f'python /kaggle/working/m8_probe.py {CFG}')
